In [ ]:
import os
from bs4 import BeautifulSoup
import pandas as pd

from bs4 import BeautifulSoup
import pandas as pd

# 1 y 2. Abrir directamente el archivo local sin usar __file__
with open("index.html", "r", encoding="utf-8") as f:
    contenido_html = f.read()

# [El resto del código hacia abajo sigue exactamente igual...]
soup = BeautifulSoup(contenido_html, "html.parser")

# 3. Pasar el contenido a BeautifulSoup para analizarlo
soup = BeautifulSoup(contenido_html, "html.parser")

# 4. Seleccionar las 20 ofertas de trabajo simuladas
# Recuerda que en tu JavaScript creamos etiquetas <article class="oferta-trabajo">
ofertas_html = soup.select("article.oferta-trabajo")

lista_extraccion = []

# 5. Extraer las variables que pide el PDF del proyecto
for oferta in ofertas_html:
    titulo = oferta.find("h2", class_="titulo-puesto").get_text().strip()
    empresa = oferta.find("div", class_="empresa").get_text().strip()
    descripcion = oferta.find("p", class_="descripcion").get_text().strip()
    salario_texto = oferta.find("p", class_="salario").get_text().strip()
    
    lista_extraccion.append({
        "Título": titulo,
        "Empresa": empresa,
        "Descripción": descripcion,
        "Salario_Original": salario_texto
    })

# 6. CREAR EL DATAFRAME DE PANDAS
df_empleos = pd.DataFrame(lista_extraccion)

# =====================================================================
# ¿QUÉ HACER CON ELLO? -> EL ANÁLISIS DE FRAUDE (Lo que pide el PDF)
# =====================================================================

# Paso A: Limpiar el salario para volverlo un número estadístico
# Quitamos "Salario: S/. ", los puntos y las comas, y quitamos " al mes"
df_empleos["Salario_Numerico"] = (
    df_empleos["Salario_Original"]
    .str.replace("Salario: S/. ", "")
    .str.replace(" al mes", "")
    .str.replace(",", "")
    .astype(float)
)

# Paso B: Aplicar un criterio estadístico/lógico para detectar fraudes
# Según los patrones, las estafas ofrecen salarios absurdamente altos por trabajos simples.
# Definimos como "Sospechosa de Fraude" cualquier oferta que pague más de S/. 5,000 al mes.
Sueldo_Maximo_Normal = 5000.0
df_empleos["¿Es_Fraude?"] = df_empleos["Salario_Numerico"] > Sueldo_Maximo_Normal

# Paso C: Separar las ofertas legítimas de las estafas para el reporte
ofertas_legitimas = df_empleos[df_empleos["¿Es_Fraude?"] == False]
ofertas_fraudulentas = df_empleos[df_empleos["¿Es_Fraude?"] == True]

# 7. Mostrar los resultados estadísticos para tu informe
print("\n=======================================================")
print("      REPORTE ESTADÍSTICO DE DETECCIÓN DE FRAUDES      ")
print("=======================================================")
print(f"Total de ofertas analizadas: {len(df_empleos)}")
print(f"✅ Ofertas Legítimas detectadas: {len(ofertas_legitimas)}")
print(f"🚨 Ofertas FRAUDULENTAS detectadas: {len(ofertas_fraudulentas)}")
print("=======================================================\n")

print("🚨 LISTA DE EMPRESAS Y PUESTOS BAJO SOSPECHA:")
for index, fila in ofertas_fraudulentas.iterrows():
    print(f"- Puesto: '{fila['Título']}' | Ofrece: {fila['Salario_Original']} | Empresa: {fila['Empresa']}")

# 8. Guardar la base de datos consolidada en un CSV para subirlo a GitHub
df_empleos.to_csv("reporte_final_empleos.csv", index=False, encoding="utf-8")

FileNotFoundError: [Errno 2] No such file or directory: 'index.html'